# Marian NMT — Build from Source on Google Colab

**Author:** [Your Name]  
**Date:** [Date]  

---

This notebook builds the Marian NMT framework from source and saves the binaries to Google Drive for reuse across sessions.

**Binaries built:**
- `marian` — training binary
- `marian-decoder` — decoding binary  
- `marian-vocab` — vocabulary builder

> ⚠️ CPU runtime is sufficient for building. GPU is not required.

## ⚡ Fast Path: Build WITH CUDA (GPU runtime) — added

The CPU-only build below (Step 1) is fine for testing, but training an RNN/Transformer NMT model on CPU is dramatically slower (hours per epoch vs minutes on GPU). If you have a GPU runtime attached (Runtime > Change runtime type > GPU) and GPU quota available, build **this** CUDA-enabled version instead — it's the same source, just without `-DCOMPILE_CUDA=OFF`. Saved separately to Drive so it doesn't overwrite your working CPU-only binaries.

> ⏳ Takes about 40-50 minutes. Uses `-j2` (not `-j$(nproc)`) to avoid the RAM-crash issue seen with full parallel builds.

In [ ]:
%%bash
# Verify GPU + CUDA toolkit are available before spending 40 min building
nvidia-smi
nvcc --version || echo 'nvcc not found - CUDA toolkit missing, this build will fail'

In [ ]:
%%bash
# Dependencies
apt-get update -qq
apt-get install -y --fix-missing cmake build-essential git libboost-all-dev 2>&1 | tail -3

# Clone into a SEPARATE folder from the CPU-only build
if [ ! -d "/content/marian-src-gpu" ]; then
    git clone https://github.com/marian-nmt/marian /content/marian-src-gpu 2>&1 | tail -3
else
    echo "Already cloned ✅"
fi

# Build WITH CUDA
cd /content/marian-src-gpu
rm -rf build && mkdir build && cd build
cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DUSE_SENTENCEPIECE=OFF \
    -DUSE_FBGEMM=OFF \
    -DCOMPILE_CUDA=ON \
    2>&1 | tail -5
make -j2 marian_train marian_decoder marian_vocab 2>&1 | tail -15
echo "=== BUILD COMPLETE ==="
ls /content/marian-src-gpu/build/marian*

### Save CUDA-enabled binaries to Google Drive (added)

Saved to a **different** Drive folder (`marian-build-gpu`) so both the CPU-only and GPU-enabled binaries are kept side by side.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
save_path = '/content/drive/MyDrive/marian-build-gpu'
os.makedirs(save_path, exist_ok=True)

for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/marian-src-gpu/build/{binary}', save_path)
    print(f'✅ {binary} saved')

print('Done — CUDA-enabled Marian saved to Google Drive 🎉')

---

## Original CPU-Only Build (kept for reference / fallback)

## Step 1: Install Dependencies, Clone and Build Marian

Install required system packages, clone the Marian repository from GitHub, and compile the source code.

- `cmake`, `build-essential`, `libboost-all-dev` — required build tools
- `-DCOMPILE_CUDA=OFF` — disable CUDA for CPU-only build
- `-j2` — use 2 parallel jobs to avoid RAM crash

> ⏳ This step takes approximately **40 minutes**.

In [ ]:
%%bash
# Dependencies
apt-get update -qq
apt-get install -y --fix-missing cmake build-essential git libboost-all-dev 2>&1 | tail -3

# Clone (skip if already exists)
if [ ! -d "/content/marian-src" ]; then
    git clone https://github.com/marian-nmt/marian /content/marian-src 2>&1 | tail -3
else
    echo "Already cloned ✅"
fi

# Build
cd /content/marian-src
rm -rf build && mkdir build && cd build
cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DUSE_SENTENCEPIECE=OFF \
    -DUSE_FBGEMM=OFF \
    -DCOMPILE_CUDA=OFF \
    2>&1 | tail -5
make -j2 marian_train marian_decoder marian_vocab 2>&1 | tail -10
echo "=== BUILD COMPLETE ==="
ls /content/marian-src/build/marian*

## Step 2: Save Marian Binaries to Google Drive

Mount Google Drive and copy the compiled binaries for permanent storage.

Binaries will be saved to: `MyDrive/marian-build/`

> In future sessions, load binaries from Drive instead of rebuilding (~5 seconds).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
save_path = '/content/drive/MyDrive/marian-build'
os.makedirs(save_path, exist_ok=True)

for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/marian-src/build/{binary}', save_path)
    print(f'✅ {binary} saved')

print('Done — Saved to Google Drive 🎉')

## Step 3: Load Marian from Google Drive (Future Sessions)

In a new Colab session, run this cell to restore Marian binaries from Drive — no rebuild needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/marian-bin', exist_ok=True)

for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/drive/MyDrive/marian-build/{binary}', '/content/marian-bin/')
    os.chmod(f'/content/marian-bin/{binary}', 0o755)
    print(f'✅ {binary}')

os.environ['PATH'] = '/content/marian-bin:' + os.environ['PATH']
print('Marian ready ✅')

### Load the CUDA-enabled binaries instead (added)

Use this cell in your training notebooks instead of the CPU-only loader above, once you've built the GPU version.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/marian-bin-gpu', exist_ok=True)

for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/drive/MyDrive/marian-build-gpu/{binary}', '/content/marian-bin-gpu/')
    os.chmod(f'/content/marian-bin-gpu/{binary}', 0o755)
    print(f'✅ {binary}')

os.environ['PATH'] = '/content/marian-bin-gpu:' + os.environ['PATH']
print('CUDA-enabled Marian ready ✅')

## Step 4: Verify Installation

Check that all Marian binaries are working correctly.

In [ ]:
!marian --version
!marian-decoder --version
!marian-vocab --version